In [21]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

In [3]:
datasets = os.listdir("../datasets/processed/")
datasets

['processed-2026-04.parquet']

In [5]:
df = pd.read_parquet(f"../datasets/processed/{datasets[-1]}")
print(df.shape)
print(df.columns)

(3831240, 5)
Index(['trip_distance', 'PULocationID', 'DOLocationID', 'fare_amount',
       'duration'],
      dtype='str')


In [17]:
X = df.drop(columns=['fare_amount'])
y = df['fare_amount']

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['fare_amount']), df['fare_amount'], test_size = 0.2, random_state = 43)

In [7]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(3064992, 4) (766248, 4) (3064992,) (766248,)


In [45]:
sc = StandardScaler()
sc.fit(X_train[['duration','trip_distance']])
X_train[['duration','trip_distance']] = sc.transform(X_train[['duration','trip_distance']])
X_test[['duration','trip_distance']] = sc.transform(X_test[['duration','trip_distance']])

In [46]:
space = {
    'n_estimators' : hp.choice('n_estimators',[25,50,75,100,125,150]),
    'max_depth' : hp.quniform('max_depth',3,10,1),
    'learning_rate' : hp.uniform('learning_rate',1e-2,3e-1),
    'objective' : 'reg:squarederror',
    'tree_method' : 'hist'
}

In [47]:
def objective(params):
    params['max_depth'] = int(params['max_depth'])

    model = XGBRegressor(**params, random_state = 42, n_jobs=-1)
    model.fit(X_train,y_train)

    preds = model.predict(X_test)

    rmse = root_mean_squared_error(preds, y_test)
    mae = mean_absolute_error(preds, y_test)
    r2 = r2_score(preds,y_test)

    return {"loss":rmse, "status":STATUS_OK}


In [48]:
trials = Trials()

best_params = fmin(
    fn = objective,
    space = space,
    algo = tpe.suggest,
    max_evals = 50,
    trials = trials,
    rstate = np.random.default_rng(42)
)

print('\nOptimization Complete!')
print("Best index choice/value found:",best_params)



100%|██████████| 50/50 [16:10<00:00, 19.41s/trial, best loss: 8.82521403883141] 

Optimization Complete!
Best index choice/value found: {'learning_rate': np.float64(0.2632967408189152), 'max_depth': np.float64(7.0), 'n_estimators': np.int64(5)}


In [49]:
n_estimators = [25,50,75,100,125,150]
n_estimators[best_params['n_estimators']]


150

In [50]:
best_params['n_estimators'] = n_estimators[best_params['n_estimators']]
print(best_params)

{'learning_rate': np.float64(0.2632967408189152), 'max_depth': np.float64(7.0), 'n_estimators': 150}


In [55]:
X_test.iloc[0,:]

trip_distance     -0.010582
PULocationID     164.000000
DOLocationID      48.000000
duration           0.083595
Name: 3618520, dtype: float64